# QuikStrike Network Explorer
ดักข้อมูลจาก QuikStrike Vol2Vol + OI Heatmap

> Jupyter ใช้ async event loop → ต้องใช้ `await`

## Cell 1 — Setup

In [3]:
from playwright.async_api import async_playwright
from pathlib import Path
import json

SESSION_FILE = Path('../data/session.json')
print('Session file exists:', SESSION_FILE.exists())

Session file exists: True


## Cell 2 — เปิด Browser

In [4]:
pw = await async_playwright().start()
browser = await pw.chromium.launch(headless=False)

if SESSION_FILE.exists():
    ctx = await browser.new_context(storage_state=str(SESSION_FILE))
    print('✅ Loaded saved session')
else:
    ctx = await browser.new_context()
    print('⚠️  No session')

page = await ctx.new_page()
print('Browser ready')

✅ Loaded saved session
Browser ready


## Cell 3 — ไปหน้า Vol2Vol แล้วดัก Response
เปลี่ยน URL ได้ตามต้องการ

In [5]:
captured = []

async def on_response(response):
    if 'quikstrike' in response.url.lower():
        try:
            body = await response.text()
        except:
            body = '(binary)'
        captured.append({
            'url': response.url,
            'method': response.request.method,
            'status': response.status,
            'content_type': response.headers.get('content-type', ''),
            'size': len(body),
            'body': body,
        })

page.on('response', on_response)

TARGET_URL = 'https://www.cmegroup.com/tools-information/quikstrike/vol2vol-expected-range.html'
# TARGET_URL = 'https://www.cmegroup.com/tools-information/quikstrike/open-interest-heatmap.html'

await page.goto(TARGET_URL, wait_until='domcontentloaded', timeout=30000)
await page.wait_for_timeout(15000)  # รอ QuikStrike โหลด
print(f'✅ Captured {len(captured)} QuikStrike responses')

✅ Captured 174 QuikStrike responses


## Cell 4 — ดู URL ทั้งหมดที่ดักได้

In [6]:
for i, r in enumerate(captured):
    print(f'[{i:2d}] {r["method"]} {r["status"]} | {r["size"]:>8,} B | {r["content_type"][:20]}')
    print(f'      {r["url"][:110]}')
    print()

[ 0] GET 200 |  171,021 B | text/html;charset=ut
      https://www.cmegroup.com/tools-information/quikstrike/vol2vol-expected-range.html

[ 1] GET 200 |    3,376 B | application/javascri
      https://www.cmegroup.com/content/dam/cmegroup/files/js/quikstrike-popup.js

[ 2] GET 301 |        8 B | text/html; charset=i
      https://www.cmegroup.com/content/cmegroup/en/subnavs/quikstrike-subnav/jcr:content.json

[ 3] GET 200 |    2,300 B | application/json
      https://www.cmegroup.com/tools-information/quikstrike/vol2vol-expected-range/jcr:content.json

[ 4] GET 200 |        2 B | application/json
      https://www.cmegroup.com/bin/service/banners.%7Ccontent%7Ccmegroup%7Cen%7Ctools-information%7Cquikstrike%7Cvol

[ 5] GET 200 |   64,832 B | application/json
      https://www.cmegroup.com/bin/service/popups.%7Ccontent%7Ccmegroup%7Cen%7Ctools-information%7Cquikstrike%7Cvol2

[ 6] GET 200 |      915 B | application/json
      https://www.cmegroup.com/subnavs/quikstrike-subnav/jcr:content.j

## Cell 5 — Filter เฉพาะ response ใหญ่ (>5KB)

In [7]:
big = sorted([r for r in captured if r['size'] > 5000], key=lambda x: -x['size'])
print(f'Large responses: {len(big)}\n')
for i, r in enumerate(big):
    print(f'[{i}] {r["size"]:>10,} B | {r["url"][:100]}')

Large responses: 180

[0]    382,010 B | https://cmegroup-tools.quikstrike.net/Scripts/Highcharts/highstock.js
[1]    382,010 B | https://cmegroup-tools.quikstrike.net/Scripts/Highcharts/highstock.js
[2]    382,010 B | https://cmegroup-tools.quikstrike.net/Scripts/Highcharts/highstock.js
[3]    382,010 B | https://cmegroup-tools.quikstrike.net/Scripts/Highcharts/highstock.js
[4]    198,633 B | https://cmegroup-tools.quikstrike.net/User/QuikStrikeView.aspx?viewitemid=IntegratedV2VExpectedRange
[5]    198,633 B | https://cmegroup-tools.quikstrike.net/User/QuikStrikeView.aspx?viewitemid=IntegratedV2VExpectedRange
[6]    198,633 B | https://cmegroup-tools.quikstrike.net/User/QuikStrikeView.aspx?viewitemid=IntegratedV2VExpectedRange
[7]    198,630 B | https://cmegroup-tools.quikstrike.net/User/QuikStrikeView.aspx?viewitemid=IntegratedV2VExpectedRange
[8]    198,624 B | https://cmegroup-tools.quikstrike.net/User/QuikStrikeView.aspx?viewitemid=IntegratedV2VExpectedRange
[9]    171,021 B | htt

## Cell 6 — ดู body ของ response ที่เลือก

In [10]:
# เปลี่ยน list และ index ได้
target_list = big   # หรือ captured
idx = 4             # index ที่อยากดู

r = target_list[idx]
print('URL:', r['url'])
print('Method:', r['method'])
print('Content-Type:', r['content_type'])
print('Size:', f"{r['size']:,} bytes")
CHAR = 5000
print(f'\n--- Body (first {CHAR} chars) ---')

if 'json' in r['content_type']:
    try:
        print(json.dumps(json.loads(r['body']), indent=2)[:CHAR])
    except:
        print(r['body'][:CHAR])
else:
    print(r['body'][:CHAR])

URL: https://cmegroup-tools.quikstrike.net/User/QuikStrikeView.aspx?viewitemid=IntegratedV2VExpectedRange&pid=25&insid=226948905&qsid=48b216a1-b0a4-4441-b00f-fbdc88c1516d
Method: GET
Content-Type: text/html; charset=utf-8
Size: 198,633 bytes

--- Body (first 5000 chars) ---


<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html xmlns="http://www.w3.org/1999/xhtml">
<head>
        
        <meta name="viewport" content='' />
    <title>

</title>

<script>
    dataLayer = [{ 'gtm.whitelist': ['google', 'k', 'customScripts'] }];
</script>

<!-- Google Tag Manager -->
<script>(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({'gtm.start':
new Date().getTime(),event:'gtm.js'});var f=d.getElementsByTagName(s)[0],
j=d.createElement(s),dl=l!='dataLayer'?'&l='+l:'';j.async=true;j.src=
'https://www.googletagmanager.com/gtm.js?id='+i+dl;f.parentNode.insertBefore(j,f);
})(window, document, 'script', 'dataLayer', 'GTM-M3HGHS6');

## Cell 7 — ดัก Request ใหม่ตอน Interact
**รัน cell นี้ก่อน** แล้วไปทำอะไรในหน้าเว็บ เช่น:
- เปลี่ยน product เป็น Gold  
- คลิก OI / Vol2Vol
- เปลี่ยน expiration

In [ ]:
new_captured = []

async def on_new(response):
    if 'quikstrike' in response.url.lower():
        try:
            body = await response.text()
        except:
            body = ''
        new_captured.append({
            'url': response.url,
            'method': response.request.method,
            'post_data': response.request.post_data,  # body ของ POST request
            'status': response.status,
            'content_type': response.headers.get('content-type', ''),
            'size': len(body),
            'body': body,
        })

page.on('response', on_new)
print('🎯 Listening... ไปทำอะไรในหน้าเว็บได้เลย')
print('แล้วรัน cell ถัดไป')

In [ ]:
# รันหลังจาก interact แล้ว
await page.wait_for_timeout(2000)

print(f'New responses: {len(new_captured)}\n')
for i, r in enumerate(new_captured):
    print(f'[{i:2d}] {r["method"]} {r["status"]} | {r["size"]:>8,} B | {r["url"][:100]}')

In [ ]:
# ดู body ของ new_captured
idx = 0  # <-- เปลี่ยน

if not new_captured:
    print('ยังไม่มี — กลับไปรัน Cell 7')
else:
    r = new_captured[idx]
    print('URL:', r['url'])
    print('Method:', r['method'])
    if r['post_data']:
        print('POST body:', r['post_data'][:500])
    print('Size:', f"{r['size']:,} bytes")
    print()
    if 'json' in r['content_type']:
        try:
            print(json.dumps(json.loads(r['body']), indent=2)[:3000])
        except:
            print(r['body'][:3000])
    else:
        print(r['body'][:3000])

## Cell 8 — Save response ที่น่าสนใจลงไฟล์

In [ ]:
# save response ที่เลือกไว้ดูทีหลัง
save_list = new_captured  # หรือ captured
idx = 0

r = save_list[idx]
out = Path('../data/raw') / 'response_debug.txt'
out.parent.mkdir(exist_ok=True)
out.write_text(f'URL: {r["url"]}\n\n{r["body"]}')
print(f'Saved to {out}')

## Cell 9 — ปิด Browser

In [ ]:
await browser.close()
await pw.stop()
print('✅ Done')